In [0]:
# ============================================================
# 05_Hiring_Trends
# Gold Layer - Job Hiring Trends Over Time
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# 1. Read Silver Clean Jobs
# ============================================================

df = spark.table("job_market.silver.jobs_clean")


# ============================================================
# 2. Prepare Date Fields
# ============================================================

hiring_base = (
    df
    .filter(F.col("posted_at").isNotNull())
    .withColumn(
        "posting_month",
        F.date_trunc("month", F.col("posted_at")).cast("date")
    )
    .withColumn(
        "posting_year",
        F.year(F.col("posted_at"))
    )
    .withColumn(
        "posting_month_number",
        F.month(F.col("posted_at"))
    )
    .withColumn(
        "posting_month_name",
        F.date_format(F.col("posted_at"), "MMMM")
    )
)


# ============================================================
# 3. Monthly Hiring Trend
# ============================================================

monthly_trends = (
    hiring_base
    .groupBy(
        "posting_month",
        "posting_year",
        "posting_month_number",
        "posting_month_name"
    )
    .agg(
        F.countDistinct("job_id").alias("job_count"),

        F.countDistinct("company").alias("company_count"),

        F.countDistinct("job_title").alias("job_title_count"),

        F.sum(
            F.when(F.col("is_remote") == True, 1).otherwise(0)
        ).alias("remote_job_count"),

        F.sum(
            F.when(F.col("workplace_type") == "hybrid", 1).otherwise(0)
        ).alias("hybrid_job_count"),

        F.sum(
            F.when(F.col("workplace_type") == "onsite", 1).otherwise(0)
        ).alias("onsite_job_count"),

        F.min("posted_at").alias("earliest_posted_at"),

        F.max("posted_at").alias("latest_posted_at")
    )
)


# ============================================================
# 4. Calculate Monthly Percentages
# ============================================================

monthly_trends = (
    monthly_trends
    .withColumn(
        "remote_percentage",
        F.round(
            F.col("remote_job_count") /
            F.col("job_count") * 100,
            2
        )
    )
    .withColumn(
        "hybrid_percentage",
        F.round(
            F.col("hybrid_job_count") /
            F.col("job_count") * 100,
            2
        )
    )
    .withColumn(
        "onsite_percentage",
        F.round(
            F.col("onsite_job_count") /
            F.col("job_count") * 100,
            2
        )
    )
)


# ============================================================
# 5. Calculate Overall Hiring Demand Percentage
# ============================================================

total_jobs = df.count()

monthly_trends = (
    monthly_trends
    .withColumn(
        "demand_percentage",
        F.round(
            F.col("job_count") / F.lit(total_jobs) * 100,
            2
        )
    )
)


# ============================================================
# 6. Month-over-Month Hiring Change
# ============================================================

trend_window = Window.orderBy("posting_month")

monthly_trends = (
    monthly_trends
    .withColumn(
        "previous_month_job_count",
        F.lag("job_count").over(trend_window)
    )
    .withColumn(
        "mom_job_change",
        F.col("job_count") -
        F.col("previous_month_job_count")
    )
    .withColumn(
        "mom_growth_percentage",
        F.when(
            F.col("previous_month_job_count") > 0,
            F.round(
                (
                    F.col("job_count") -
                    F.col("previous_month_job_count")
                )
                /
                F.col("previous_month_job_count")
                * 100,
                2
            )
        )
        .otherwise(None)
    )
)


# ============================================================
# 7. Add Hiring Trend Classification
# ============================================================

monthly_trends = (
    monthly_trends
    .withColumn(
        "hiring_trend",
        F.when(
            F.col("mom_growth_percentage") > 5,
            "Growing"
        )
        .when(
            F.col("mom_growth_percentage") < -5,
            "Declining"
        )
        .when(
            F.col("mom_growth_percentage").isNotNull(),
            "Stable"
        )
        .otherwise("Baseline")
    )
)


# ============================================================
# 8. Final Gold Schema
# ============================================================

hiring_trends = (
    monthly_trends
    .select(
        "posting_month",
        "posting_year",
        "posting_month_number",
        "posting_month_name",

        "job_count",
        "company_count",
        "job_title_count",

        "remote_job_count",
        "hybrid_job_count",
        "onsite_job_count",

        "remote_percentage",
        "hybrid_percentage",
        "onsite_percentage",

        "demand_percentage",

        "previous_month_job_count",
        "mom_job_change",
        "mom_growth_percentage",
        "hiring_trend",

        "earliest_posted_at",
        "latest_posted_at"
    )
    .orderBy("posting_month")
)


# ============================================================
# 9. Validation
# ============================================================

print("Total Silver jobs:", total_jobs)

print(
    "Total jobs represented in Hiring Trends:",
    hiring_trends.select(
        F.sum("job_count").alias("total")
    ).collect()[0]["total"]
)

display(hiring_trends)


# ============================================================
# 10. Write Gold Table
# ============================================================

(
    hiring_trends
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("job_market.gold.hiring_trends")
)


# ============================================================
# 11. Final Verification
# ============================================================

print("Gold table created successfully:")
print("job_market.gold.hiring_trends")

display(
    spark.table("job_market.gold.hiring_trends")
    .orderBy("posting_month")
)

Total Silver jobs: 2221


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total jobs represented in Hiring Trends: 2221


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


posting_month,posting_year,posting_month_number,posting_month_name,job_count,company_count,job_title_count,remote_job_count,hybrid_job_count,onsite_job_count,remote_percentage,hybrid_percentage,onsite_percentage,demand_percentage,previous_month_job_count,mom_job_change,mom_growth_percentage,hiring_trend,earliest_posted_at,latest_posted_at
2009-12-01,2009,12,December,1,1,1,0,1,0,0.0,100.0,0.0,0.05,null,null,null,Baseline,2009-12-05T00:00:00.000Z,2009-12-05T00:00:00.000Z
2013-01-01,2013,1,January,1,1,1,0,1,0,0.0,100.0,0.0,0.05,1,0,0.0,Stable,2013-01-31T00:00:00.000Z,2013-01-31T00:00:00.000Z
2013-03-01,2013,3,March,1,1,1,0,1,0,0.0,100.0,0.0,0.05,1,0,0.0,Stable,2013-03-08T00:00:00.000Z,2013-03-08T00:00:00.000Z
2014-01-01,2014,1,January,1,1,1,0,1,0,0.0,100.0,0.0,0.05,1,0,0.0,Stable,2014-01-23T00:00:00.000Z,2014-01-23T00:00:00.000Z
2016-02-01,2016,2,February,2,2,2,0,1,1,0.0,50.0,50.0,0.09,1,1,100.0,Growing,2016-02-08T19:02:13.000Z,2016-02-24T23:37:05.000Z
2016-04-01,2016,4,April,1,1,1,0,1,0,0.0,100.0,0.0,0.05,2,-1,-50.0,Declining,2016-04-08T15:51:10.000Z,2016-04-08T15:51:10.000Z
2016-10-01,2016,10,October,1,1,1,0,0,1,0.0,0.0,100.0,0.05,1,0,0.0,Stable,2016-10-06T22:34:11.000Z,2016-10-06T22:34:11.000Z
2016-12-01,2016,12,December,1,1,1,0,1,0,0.0,100.0,0.0,0.05,1,0,0.0,Stable,2016-12-08T22:18:29.000Z,2016-12-08T22:18:29.000Z
2017-08-01,2017,8,August,1,1,1,0,1,0,0.0,100.0,0.0,0.05,1,0,0.0,Stable,2017-08-10T19:39:38.000Z,2017-08-10T19:39:38.000Z
2018-04-01,2018,4,April,1,1,1,0,0,1,0.0,0.0,100.0,0.05,1,0,0.0,Stable,2018-04-24T16:45:55.000Z,2018-04-24T16:45:55.000Z


Gold table created successfully:
job_market.gold.hiring_trends


posting_month,posting_year,posting_month_number,posting_month_name,job_count,company_count,job_title_count,remote_job_count,hybrid_job_count,onsite_job_count,remote_percentage,hybrid_percentage,onsite_percentage,demand_percentage,previous_month_job_count,mom_job_change,mom_growth_percentage,hiring_trend,earliest_posted_at,latest_posted_at
2009-12-01,2009,12,December,1,1,1,0,1,0,0.0,100.0,0.0,0.05,null,null,null,Baseline,2009-12-05T00:00:00.000Z,2009-12-05T00:00:00.000Z
2013-01-01,2013,1,January,1,1,1,0,1,0,0.0,100.0,0.0,0.05,1,0,0.0,Stable,2013-01-31T00:00:00.000Z,2013-01-31T00:00:00.000Z
2013-03-01,2013,3,March,1,1,1,0,1,0,0.0,100.0,0.0,0.05,1,0,0.0,Stable,2013-03-08T00:00:00.000Z,2013-03-08T00:00:00.000Z
2014-01-01,2014,1,January,1,1,1,0,1,0,0.0,100.0,0.0,0.05,1,0,0.0,Stable,2014-01-23T00:00:00.000Z,2014-01-23T00:00:00.000Z
2016-02-01,2016,2,February,2,2,2,0,1,1,0.0,50.0,50.0,0.09,1,1,100.0,Growing,2016-02-08T19:02:13.000Z,2016-02-24T23:37:05.000Z
2016-04-01,2016,4,April,1,1,1,0,1,0,0.0,100.0,0.0,0.05,2,-1,-50.0,Declining,2016-04-08T15:51:10.000Z,2016-04-08T15:51:10.000Z
2016-10-01,2016,10,October,1,1,1,0,0,1,0.0,0.0,100.0,0.05,1,0,0.0,Stable,2016-10-06T22:34:11.000Z,2016-10-06T22:34:11.000Z
2016-12-01,2016,12,December,1,1,1,0,1,0,0.0,100.0,0.0,0.05,1,0,0.0,Stable,2016-12-08T22:18:29.000Z,2016-12-08T22:18:29.000Z
2017-08-01,2017,8,August,1,1,1,0,1,0,0.0,100.0,0.0,0.05,1,0,0.0,Stable,2017-08-10T19:39:38.000Z,2017-08-10T19:39:38.000Z
2018-04-01,2018,4,April,1,1,1,0,0,1,0.0,0.0,100.0,0.05,1,0,0.0,Stable,2018-04-24T16:45:55.000Z,2018-04-24T16:45:55.000Z


In [0]:
total_jobs = df.count()

In [0]:
total_jobs = df.count()

In [0]:
total_jobs = df.filter(F.col("posted_at").isNotNull()).count()

In [0]:
total_jobs = df.filter(F.col("posted_at").isNotNull()).count()

In [0]:
display(
    spark.table("job_market.gold.hiring_trends")
    .orderBy("posting_month")
)

posting_month,posting_year,posting_month_number,posting_month_name,job_count,company_count,job_title_count,remote_job_count,hybrid_job_count,onsite_job_count,remote_percentage,hybrid_percentage,onsite_percentage,demand_percentage,previous_month_job_count,mom_job_change,mom_growth_percentage,hiring_trend,earliest_posted_at,latest_posted_at
2009-12-01,2009,12,December,1,1,1,0,1,0,0.0,100.0,0.0,0.05,null,null,null,Baseline,2009-12-05T00:00:00.000Z,2009-12-05T00:00:00.000Z
2013-01-01,2013,1,January,1,1,1,0,1,0,0.0,100.0,0.0,0.05,1,0,0.0,Stable,2013-01-31T00:00:00.000Z,2013-01-31T00:00:00.000Z
2013-03-01,2013,3,March,1,1,1,0,1,0,0.0,100.0,0.0,0.05,1,0,0.0,Stable,2013-03-08T00:00:00.000Z,2013-03-08T00:00:00.000Z
2014-01-01,2014,1,January,1,1,1,0,1,0,0.0,100.0,0.0,0.05,1,0,0.0,Stable,2014-01-23T00:00:00.000Z,2014-01-23T00:00:00.000Z
2016-02-01,2016,2,February,2,2,2,0,1,1,0.0,50.0,50.0,0.09,1,1,100.0,Growing,2016-02-08T19:02:13.000Z,2016-02-24T23:37:05.000Z
2016-04-01,2016,4,April,1,1,1,0,1,0,0.0,100.0,0.0,0.05,2,-1,-50.0,Declining,2016-04-08T15:51:10.000Z,2016-04-08T15:51:10.000Z
2016-10-01,2016,10,October,1,1,1,0,0,1,0.0,0.0,100.0,0.05,1,0,0.0,Stable,2016-10-06T22:34:11.000Z,2016-10-06T22:34:11.000Z
2016-12-01,2016,12,December,1,1,1,0,1,0,0.0,100.0,0.0,0.05,1,0,0.0,Stable,2016-12-08T22:18:29.000Z,2016-12-08T22:18:29.000Z
2017-08-01,2017,8,August,1,1,1,0,1,0,0.0,100.0,0.0,0.05,1,0,0.0,Stable,2017-08-10T19:39:38.000Z,2017-08-10T19:39:38.000Z
2018-04-01,2018,4,April,1,1,1,0,0,1,0.0,0.0,100.0,0.05,1,0,0.0,Stable,2018-04-24T16:45:55.000Z,2018-04-24T16:45:55.000Z
